# Viz Utils Smoke Test

Notebook này dùng để kiểm tra nhanh module `srcs/00_utils/viz_utils.py` trên dataset `ml_mart.base` đã export ra Parquet.

Mặc định notebook đọc:

```text
data/mlmart_base/v1_preprocessing.parquet
```

Nếu chạy trên Google Colab hoặc đổi vị trí file, chỉ cần sửa biến `DATA_PATH`.

Nguyên tắc:

- Notebook chỉ đọc file local, không ghi database.
- Các hàm plot trả về `fig, ax`, không tự `plt.show()`.
- Có thể đổi `VALUE_COL` từ `energy_generated_kwh` sang `energy_pred`, `residual`, hoặc cột khác.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """Find repo root from notebook cwd or Colab working directory."""
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "srcs" / "00_utils" / "viz_utils.py").exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy srcs/00_utils/viz_utils.py. "
        "Hãy chạy notebook trong repo root hoặc copy folder srcs lên Colab."
    )

PROJECT_ROOT = find_project_root()
UTILS_DIR = PROJECT_ROOT / "srcs" / "00_utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("UTILS_DIR    =", UTILS_DIR)

from viz_utils import (
    plot_actual_vs_pred,
    plot_correlation_heatmap,
    plot_distribution,
    plot_error_by_temperature_bin,
    plot_global_overview,
    plot_heatmap_site_hour,
    plot_local_zoom,
    plot_seasonal_profile,
    plot_temperature_relationship,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200


## 1. Đọc dữ liệu

Sửa `DATA_PATH` nếu file Parquet nằm chỗ khác.


In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "mlmart_base" / "v1_preprocessing.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_PATH}. "
        "Hãy export ml_mart.base trước hoặc sửa DATA_PATH trong notebook."
    )

df = pd.read_parquet(DATA_PATH)
print(df.shape)
df.head()


## 2. Chuẩn hóa tên cột dùng trong notebook

Notebook không sửa file gốc, chỉ tạo vài cột phụ trong RAM nếu cần để demo hàm sau training.


In [ ]:
TIMESTAMP_COL = "timestamp"
SITE_COL = "site_id" if "site_id" in df.columns else ("sitekey" if "sitekey" in df.columns else "site_key")
VALUE_COL = "energy_generated_kwh"
TEMP_COL = "temperature_c"

df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce")
df = df.dropna(subset=[TIMESTAMP_COL, SITE_COL, VALUE_COL]).copy()

# Demo after-training plots: nếu chưa có prediction thì tạo placeholder đơn giản bằng rolling mean theo site.
# Khi có model thật, thay energy_pred bằng output model thật.
if "energy_actual" not in df.columns:
    df["energy_actual"] = df[VALUE_COL]

if "energy_pred" not in df.columns:
    df = df.sort_values([SITE_COL, TIMESTAMP_COL])
    # Placeholder chỉ để test visualization: centered rolling giúp đường cam
    # đi sát đường xanh hơn và không bị trễ pha như trailing rolling.
    # Không dùng cách này làm baseline model thật vì center=True nhìn cả tương lai.
    df["energy_pred"] = (
        df.groupby(SITE_COL, group_keys=False)[VALUE_COL]
        .transform(lambda s: s.rolling(window=5, min_periods=1, center=True).mean())
    )

df["residual"] = df["energy_actual"] - df["energy_pred"]

print("rows:", len(df))
print("sites:", df[SITE_COL].nunique())
print("time range:", df[TIMESTAMP_COL].min(), "->", df[TIMESTAMP_COL].max())
print("site col:", SITE_COL)
print("value col:", VALUE_COL)


## 3. Chọn một site và một khoảng local để zoom


In [ ]:
SITE_ID = df[SITE_COL].dropna().iloc[0]
N_DAYS = 3

# Chọn ngày bắt đầu theo chính site đang xem, không dùng min global.
site_df = df[df[SITE_COL].astype(str) == str(SITE_ID)].sort_values(TIMESTAMP_COL)
START_DATE = site_df[TIMESTAMP_COL].min().normalize()
END_DATE = START_DATE + pd.Timedelta(days=N_DAYS)
local_rows = site_df[(site_df[TIMESTAMP_COL] >= START_DATE) & (site_df[TIMESTAMP_COL] < END_DATE)]

print("SITE_ID:", SITE_ID)
print("START_DATE:", START_DATE)
print("END_DATE:", END_DATE)
print("local rows:", len(local_rows))
print("local min/max:", local_rows[TIMESTAMP_COL].min(), "->", local_rows[TIMESTAMP_COL].max())
print("unique dates:", local_rows[TIMESTAMP_COL].dt.date.nunique())


## 4. Plot global overview


In [ ]:
fig, ax = plot_global_overview(
    df,
    value_col=VALUE_COL,
    freq="ME",
    timestamp_col=TIMESTAMP_COL,
    agg="mean",
)
plt.show()


## 5. Seasonal profile cho một trạm


In [ ]:
fig, ax = plot_seasonal_profile(
    df,
    site_id=SITE_ID,
    value_col=VALUE_COL,
    timestamp_col=TIMESTAMP_COL,
    site_col=SITE_COL,
)
plt.show()


## 6. Local zoom theo độ phân giải gốc


In [ ]:
fig, ax = plot_local_zoom(
    df,
    site_id=SITE_ID,
    start_date=START_DATE,
    n_days=N_DAYS,
    value_col=VALUE_COL,
    timestamp_col=TIMESTAMP_COL,
    site_col=SITE_COL,
)
plt.show()


## 7. Actual vs prediction + residual

Dùng sau training. Hiện nếu chưa có `energy_pred`, notebook đã tạo placeholder rolling mean để test hàm.


In [ ]:
local_pred = df[(df[SITE_COL].astype(str) == str(SITE_ID)) & (df[TIMESTAMP_COL] >= START_DATE) & (df[TIMESTAMP_COL] < END_DATE)].copy()
local_pred["residual"] = local_pred["energy_actual"] - local_pred["energy_pred"]

res = local_pred["residual"].dropna()
print("Residual diagnostics trong local window")
print("rows       :", len(local_pred))
print("min        :", res.min())
print("max        :", res.max())
print("mean       :", res.mean())
print("MAE        :", res.abs().mean())
print("RMSE       :", (res.pow(2).mean()) ** 0.5)
print("P95 |err|  :", res.abs().quantile(0.95))
print("P99 |err|  :", res.abs().quantile(0.99))


In [ ]:
fig, axes = plot_actual_vs_pred(
    df,
    site_id=SITE_ID,
    start_date=START_DATE,
    n_days=N_DAYS,
    actual_col="energy_actual",
    pred_col="energy_pred",
    timestamp_col=TIMESTAMP_COL,
    site_col=SITE_COL,
)
plt.show()


## 8. Temperature relationship


In [ ]:
if TEMP_COL in df.columns:
    fig, ax = plot_temperature_relationship(
        df,
        value_col=VALUE_COL,
        temp_col=TEMP_COL,
        sample_size=50_000,
    )
    plt.show()
else:
    print(f"Skip: không có cột {TEMP_COL}")


## 9. Error by temperature bin


In [ ]:
if TEMP_COL in df.columns:
    fig, ax = plot_error_by_temperature_bin(
        df,
        y_true="energy_actual",
        y_pred="energy_pred",
        temp_col=TEMP_COL,
        bins=10,
    )
    plt.show()
else:
    print(f"Skip: không có cột {TEMP_COL}")


## 10. Distribution plot


In [ ]:
fig, ax = plot_distribution(
    df,
    value_col=VALUE_COL,
    by=None,
    bins=80,
    sample_size=200_000,
)
plt.show()

fig, ax = plot_distribution(
    df,
    value_col="residual",
    by=None,
    bins=80,
    sample_size=200_000,
)
plt.show()


## 11. Heatmap site × hour


In [ ]:
fig, ax = plot_heatmap_site_hour(
    df,
    value_col=VALUE_COL,
    agg="mean",
    timestamp_col=TIMESTAMP_COL,
    site_col=SITE_COL,
)
plt.show()

if "gmm_if_outlier_flag" in df.columns:
    fig, ax = plot_heatmap_site_hour(
        df.assign(gmm_if_outlier_flag=df["gmm_if_outlier_flag"].astype(int)),
        value_col="gmm_if_outlier_flag",
        agg="sum",
        timestamp_col=TIMESTAMP_COL,
        site_col=SITE_COL,
    )
    plt.show()


## 12. Correlation heatmap


In [ ]:
candidate_cols = [
    VALUE_COL,
    "energy_pred",
    "residual",
    "temperature_c",
    "shortwave_radiation",
    "direct_normal_irradiance",
    "diffuse_solar_radiation",
    "cloud_cover_total",
    "wind_speed",
    "precipitation_mm",
    "sunshine_duration",
]
cols = [col for col in candidate_cols if col in df.columns]

fig, ax = plot_correlation_heatmap(df, cols=cols, method="pearson", annotate=True)
plt.show()


## 13. Lưu hình nếu cần

Uncomment block dưới đây nếu muốn export ảnh ra folder `pictures/viz_utils_smoke_test`.


In [ ]:
# OUTPUT_DIR = PROJECT_ROOT / "pictures" / "viz_utils_smoke_test"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
# fig, ax = plot_global_overview(df, value_col=VALUE_COL, freq="ME", timestamp_col=TIMESTAMP_COL)
# fig.savefig(OUTPUT_DIR / "01_global_overview.png", bbox_inches="tight")
# plt.close(fig)
#
# fig, ax = plot_heatmap_site_hour(df, value_col=VALUE_COL, timestamp_col=TIMESTAMP_COL, site_col=SITE_COL)
# fig.savefig(OUTPUT_DIR / "02_site_hour_heatmap.png", bbox_inches="tight")
# plt.close(fig)
